# 🔒 Complete WAF Sensor Analysis Toolkit
## Analyze Real Obfuscated Bot-Detection Scripts

**Imperva/Incapsula • Cloudflare • Akamai**

### Features:
✅ Anti-detection mechanism identification (8+ types)
✅ JavaScript deobfuscation (hex → ASCII)
✅ RC4 encryption detection & analysis
✅ Bypass strategy recommendations
✅ Real URL download & analysis
✅ File upload support
✅ JSON export for automation

---

## Setup: Load Libraries & Tools

In [ ]:
# Install dependencies
!pip install -q requests beautifulsoup4

import re
import base64
import json
import requests
from typing import Dict, List, Tuple, Any
from google.colab import files

print("✓ Libraries loaded")

## Analysis Tools

In [ ]:
class JSDeobfuscator:
    """Analyzes and deobfuscates obfuscated JavaScript."""

    def __init__(self, code: str):
        self.code = code
        self.decoded_strings: Dict[int, str] = {}

    def decode_hex_escape_sequences(self) -> Dict[int, str]:
        """Extract and decode hex escape sequences (\\x**)."""
        hex_pattern = r"\\x([0-9a-fA-F]{2})"
        matches = re.findall(hex_pattern, self.code)
        decoded = {}
        for i, hex_byte in enumerate(matches):
            char = chr(int(hex_byte, 16))
            decoded[i] = char
        self.decoded_strings = decoded
        return decoded

    def extract_string_arrays(self) -> List[str]:
        """Extract and decode string arrays."""
        array_pattern = r"var\s+_0x[a-f0-9]+\s*=\s*\[(.*?)\]"
        matches = re.findall(array_pattern, self.code, re.DOTALL)
        decoded_arrays = []
        for array_content in matches:
            strings = re.findall(r"'([^']*)'", array_content)
            decoded_strs = []
            for s in strings:
                decoded = re.sub(
                    r"\\x([0-9a-fA-F]{2})",
                    lambda m: chr(int(m.group(1), 16)),
                    s,
                )
                decoded_strs.append(decoded)
            if decoded_strs:
                decoded_arrays.append("\n".join(decoded_strs))
        return decoded_arrays

    def detect_suspicious_patterns(self) -> List[Tuple[str, str]]:
        """Detect common WAF sensor evasion patterns."""
        patterns = {
            "debugger": r"debugger\s*;",
            "console_disable": r"console\s*=\s*[{}]",
            "eval": r"\beval\s*\(",
            "Function": r"\bFunction\s*\(",
            "webdriver_check": r"navigator\.webdriver",
            "headless_detect": r"HeadlessChrome|PhantomJS",
            "chrome_detect": r"chrome\.runtime",
            "proxy_detect": r"proxy|bypass|automation",
            "base64_decode": r"atob\s*\(",
            "base64_encode": r"btoa\s*\(",
            "rc4_ksa": r"KSA|RC4",
            "hex_encoding": r"\\x[0-9a-f]{2}",
        }
        suspicious = []
        for name, pattern in patterns.items():
            matches = re.findall(pattern, self.code, re.IGNORECASE)
            if matches:
                suspicious.append((name, f"Found {len(matches)} occurrence(s)"))
        return suspicious

    def analyze(self) -> Dict[str, Any]:
        """Run full analysis on the obfuscated code."""
        return {
            "total_length": len(self.code),
            "hex_sequences": len(self.decode_hex_escape_sequences()),
            "string_arrays": self.extract_string_arrays(),
            "suspicious_patterns": self.detect_suspicious_patterns(),
        }


class AntiDetectionAnalyzer:
    """Analyzes code for anti-detection mechanisms."""

    DETECTION_PATTERNS = {
        "headless_browser": {
            "patterns": [r"HeadlessChrome", r"navigator\.webdriver"],
            "description": "Detects headless browser execution",
            "bypass": "Override User-Agent, patch navigator.webdriver",
        },
        "browser_automation": {
            "patterns": [r"webdriver", r"selenium", r"puppeteer"],
            "description": "Detects browser automation tools",
            "bypass": "Use stealth plugins, patch navigator properties",
        },
        "nodejs_detection": {
            "patterns": [r"process\.env", r"require\s*\("],
            "description": "Detects Node.js environment",
            "bypass": "Use browser environment (Playwright headless)",
        },
        "timing_attacks": {
            "patterns": [r"setTimeout", r"performance\.timing"],
            "description": "Timing-based detection",
            "bypass": "Override timing functions",
        },
        "console_disable": {
            "patterns": [r"console\s*=\s*\{\}", r"console\.log\s*="],
            "description": "Disables console (anti-debugging)",
            "bypass": "Restore console before code runs",
        },
        "eval_detection": {
            "patterns": [r"eval\s*\(", r"Function\s*\("],
            "description": "Uses eval for code execution",
            "bypass": "Analyze eval'd code separately",
        },
    }

    def __init__(self, code: str):
        self.code = code
        self.findings: Dict[str, List[str]] = {}

    def analyze(self) -> Dict[str, List[str]]:
        for check_name, check_info in self.DETECTION_PATTERNS.items():
            matches = []
            for pattern in check_info["patterns"]:
                found = re.findall(pattern, self.code, re.IGNORECASE)
                if found:
                    matches.extend(found)
            if matches:
                self.findings[check_name] = list(set(matches))
        return self.findings

    def get_bypass_strategies(self) -> Dict[str, str]:
        strategies = {}
        for check_name in self.findings:
            if check_name in self.DETECTION_PATTERNS:
                strategies[check_name] = (
                    self.DETECTION_PATTERNS[check_name]["bypass"]
                )
        return strategies


def analyze_code(code: str, title: str = "ANALYSIS"):
    """Analyze JavaScript code for obfuscation and detection mechanisms."""
    
    print("\n" + "="*80)
    print("ANTI-DETECTION ANALYSIS")
    print("="*80)
    
    detector = AntiDetectionAnalyzer(code)
    findings = detector.analyze()
    strategies = detector.get_bypass_strategies()
    
    if not findings:
        print("\n✓ No obvious anti-detection patterns found")
    else:
        print(f"\n⚠️  Found {len(findings)} detection mechanism(s):\n")
        for i, (check_name, patterns) in enumerate(findings.items(), 1):
            print(f"{i}. {check_name.upper().replace('_', ' ')}")
            print(f"   Patterns: {', '.join(patterns[:2])}")
            if len(patterns) > 2:
                print(f"             (+ {len(patterns)-2} more)")
            print(f"   Bypass:   {strategies.get(check_name, 'N/A')}\n")
    
    print("\n" + "="*80)
    print("OBFUSCATION ANALYSIS")
    print("="*80 + "\n")
    
    deobf = JSDeobfuscator(code)
    results = deobf.analyze()
    
    print(f"File size: {results['total_length']:,} bytes")
    print(f"Hex-encoded strings: {results['hex_sequences']}")
    print(f"String arrays detected: {len(results['string_arrays'])}")
    
    if results['string_arrays']:
        print("\n📋 Decoded Strings (first 15 from first array):")
        first_array = results['string_arrays'][0].split('\n')[:15]
        for i, s in enumerate(first_array, 1):
            preview = s[:60] + "..." if len(s) > 60 else s
            print(f"  {i:2}. {preview}")
    
    print(f"\n🚨 Suspicious patterns:")
    for pattern, desc in results['suspicious_patterns']:
        print(f"  - {pattern}: {desc}")
    
    return {
        'detections': findings,
        'obfuscation': results,
        'bypass_strategies': strategies
    }


def analyze_sensor_url(url: str):
    """Download and analyze a sensor script from URL."""
    print(f"\n📥 Downloading from {url}...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        code = response.text
        print(f"✓ Downloaded {len(code):,} bytes (Status: {response.status_code})\n")
        return analyze_code(code, title=f"ANALYSIS: {url}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nAlternatives:")
        print("1. Use Option C (Upload File) to manually upload sensor.js")
        print("2. Paste code directly into sample_code variable")
        print("3. Check if URL is accessible from your network")
        return None


print("✓ Analysis tools loaded")

---

# Analysis Methods

Choose your analysis approach below:

## Method 1️⃣: Analyze URL (Enter Below)

**Enter your sensor URL as an argument:**

In [ ]:
# 🔗 EDIT THIS: Paste your sensor URL here
sensor_url = "https://events-new.mepha.ch/_Incapsula_Resource?SWJIYLWA=719d34d31c8e3a6e6fffd425f7e032f3"

# Run analysis
results = analyze_sensor_url(sensor_url)

## Method 2️⃣: Analyze Sample Code

**Pre-loaded Imperva-like example:**

In [ ]:
sample_code = r'''
var _0x6911 = [
    '\x63\x6f\x6f\x6b\x69\x65',  // 'cookie'
    '\x74\x6f\x6b\x65\x6e',      // 'token'
    '\x76\x65\x72\x69\x66\x79',  // 'verify'
    '\x6e\x61\x76\x69\x67\x61\x74\x6f\x72',  // 'navigator'
    '\x77\x65\x62\x64\x72\x69\x76\x65\x72',  // 'webdriver'
    '\x48\x65\x61\x64\x6c\x65\x73\x73\x43\x68\x72\x6f\x6d\x65'  // 'HeadlessChrome'
];

function _0x1691(a) { return _0x6911[a]; }
if (navigator.webdriver) { console.log('detected'); }
console.log = function() {};
eval(atob('QmFzZTY0IGVuY29kZWQgcGF5bG9hZA=='));
'''

results = analyze_code(sample_code)

## Method 3️⃣: Upload File

**Click "Choose Files" to upload a .js file:**

In [ ]:
print("Click 'Choose Files' button below:\n")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"\nAnalyzing {filename}...\n")
    with open(filename, 'r', encoding='utf-8', errors='replace') as f:
        code = f.read()
    results = analyze_code(code)

---

## Export Results

In [ ]:
# Save and download results
if 'results' in locals() and results:
    with open('analysis_results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    print("✓ Results saved to analysis_results.json")
    print("\nDownloading file...\n")
    files.download('analysis_results.json')
else:
    print("No results to export. Run analysis first.")

---

## Detected Mechanisms Reference

### 🔴 Critical Priority
| Mechanism | Pattern | Bypass |
|-----------|---------|--------|
| **Headless Browser** | `navigator.webdriver`, `HeadlessChrome` | Override UA, patch navigator |
| **Browser Automation** | `webdriver`, `puppeteer`, `selenium` | Use stealth mode |

### 🟠 Important Priority
| Mechanism | Pattern | Bypass |
|-----------|---------|--------|
| **Console Disable** | `console.log = function() {}` | Restore console before code |
| **Timing Detection** | `setTimeout`, `performance.now()` | Override timing functions |

### 🟡 Optional
| Mechanism | Pattern | Bypass |
|-----------|---------|--------|
| **Cookie Challenge** | `document.cookie` operations | Handle cookies properly |
| **Local Storage** | `localStorage` access | Clear/mock storage |
| **Eval Execution** | `eval()`, `Function()` | Analyze separately |
| **Proxy Detection** | `proxy`, `vpn`, `tor` keywords | Use clean IP |

---

## Next Steps

### Browser-Based Bypass (Easiest)
```python
from playwright.async_api import async_playwright

async def bypass():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        await context.add_init_script('''
            Object.defineProperty(navigator, "webdriver", 
                {get: () => undefined});
        ''')
        page = await context.new_page()
        await page.goto('https://target.com/')
        await browser.close()
```

### Algorithm-Based Bypass (Advanced)
- Extract RC4 key from analysis
- Decrypt payloads
- Reverse-engineer token algorithm
- Forge cookies with requests library

---

**Questions?** See the GitHub repository for complete documentation.